In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Make project-level src modules importable when the notebook is run from notebooks/.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import load_data, clean_data

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
df = load_data("../data/marketing_AB.csv")

print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Dataset loaded successfully.
Rows: 588,101
Columns: 7


In [3]:
df_clean = clean_data(df)

print("Approved cleaning steps applied successfully.")
print(f"Current columns: {df_clean.columns.tolist()}")


Approved cleaning steps applied successfully.
Current columns: ['user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


In [4]:
print("Exported index removal verified.")
print("'Unnamed: 0' present:", "Unnamed: 0" in df_clean.columns)


Exported index removal verified.
'Unnamed: 0' present: False


In [5]:
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]}")
print(df_clean.head())

Rows: 588,101
Columns: 6
   user id test group  converted  total ads most ads day  most ads hour
0  1069124         ad      False        130       Monday             20
1  1119715         ad      False         93      Tuesday             22
2  1144181         ad      False         21      Tuesday             18
3  1435133         ad      False        355      Tuesday             10
4  1015700         ad      False        276       Friday             14


In [6]:
df_clean.dtypes

user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object

In [7]:
print("Data type validation:")
print(f"user id       : {df_clean['user id'].dtype}")
print(f"test group    : {df_clean['test group'].dtype}")
print(f"converted     : {df_clean['converted'].dtype}")
print(f"total ads     : {df_clean['total ads'].dtype}")
print(f"most ads day  : {df_clean['most ads day'].dtype}")
print(f"most ads hour : {df_clean['most ads hour'].dtype}")

Data type validation:
user id       : int64
test group    : object
converted     : bool
total ads     : int64
most ads day  : object
most ads hour : int64


In [8]:
expected_groups = {"ad", "psa"}

actual_groups = set(df_clean["test group"].unique())

print("Test group values:", actual_groups)
print("Unexpected groups:", actual_groups - expected_groups)

Test group values: {'psa', 'ad'}
Unexpected groups: set()


In [9]:
expected_days = {
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
}

actual_days = set(df_clean["most ads day"].unique())

print("Most ads day values:", actual_days)
print("Unexpected days:", actual_days - expected_days)

Most ads day values: {'Friday', 'Tuesday', 'Wednesday', 'Saturday', 'Thursday', 'Sunday', 'Monday'}
Unexpected days: set()


In [10]:
expected_converted = {True, False}

actual_converted = set(df_clean["converted"].unique())

print("Converted values:", actual_converted)
print("Unexpected values:", actual_converted - expected_converted)

Converted values: {np.False_, np.True_}
Unexpected values: set()


In [11]:
numeric_checks = pd.DataFrame({
    "min": [
        df_clean["total ads"].min(),
        df_clean["most ads hour"].min()
    ],
    "max": [
        df_clean["total ads"].max(),
        df_clean["most ads hour"].max()
    ],
    "missing": [
        df_clean["total ads"].isna().sum(),
        df_clean["most ads hour"].isna().sum()
    ]
}, index=["total_ads", "most_ads_hour"])

numeric_checks

,min,max,missing
total_ads,1,2065,0
most_ads_hour,0,23,0


In [12]:
print("Invalid total_ads values:", (df_clean["total ads"] <= 0).sum())
print("Invalid hours below 0:", (df_clean["most ads hour"] < 0).sum())
print("Invalid hours above 23:", (df_clean["most ads hour"] > 23).sum())

Invalid total_ads values: 0
Invalid hours below 0: 0
Invalid hours above 23: 0


In [13]:
exposure_percentiles = df_clean["total ads"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
)

exposure_percentiles

0.900      57.0
0.950      88.0
0.990     202.0
0.995     276.0
0.999     499.9
1.000    2065.0
Name: total ads, dtype: float64

In [14]:
thresholds = [100, 200, 500, 1000, 1500, 2000]

for threshold in thresholds:
    count = (df_clean["total ads"] > threshold).sum()
    percentage = count / len(df_clean) * 100
    
    print(
        f"> {threshold:4} ads: "
        f"{count:6,} users ({percentage:.4f}%)"
    )

>  100 ads: 23,064 users (3.9218%)
>  200 ads:  5,952 users (1.0121%)
>  500 ads:    585 users (0.0995%)
> 1000 ads:     36 users (0.0061%)
> 1500 ads:      4 users (0.0007%)
> 2000 ads:      1 users (0.0002%)


In [15]:
print("Original shape:", df.shape)
print("Cleaned shape :", df_clean.shape)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nUnique users:", df_clean["user id"].nunique())
print("Total rows:", len(df_clean))

print("\nTest groups:")
print(df_clean["test group"].value_counts())

print("\nConversion values:")
print(df_clean["converted"].value_counts())

Original shape: (588101, 7)
Cleaned shape : (588101, 6)

Missing values:
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

Unique users: 588101
Total rows: 588101

Test groups:
test group
ad     564577
psa     23524
Name: count, dtype: int64

Conversion values:
converted
False    573258
True      14843
Name: count, dtype: int64


In [16]:
cleaned_path = "../data/marketing_AB_clean.csv"

df_clean.to_csv(cleaned_path, index=False)

print(f"Cleaned dataset saved to: {cleaned_path}")

Cleaned dataset saved to: ../data/marketing_AB_clean.csv


In [17]:
df_check = pd.read_csv(cleaned_path)

print(f"Reloaded dataset shape: {df_check.shape}")
print(f"Columns: {df_check.columns.tolist()}")

Reloaded dataset shape: (588101, 6)
Columns: ['user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


## Cleaning Summary

### Changes made

- Removed `Unnamed: 0`, which was confirmed to be a sequential exported index.
- No missing values required treatment.
- No duplicate `user_id` values were found.
- No invalid treatment groups, conversion values, days, or hours were found.
- No invalid `total_ads` values were found.
- Extreme `total_ads` observations were investigated and retained because no evidence indicated that they were erroneous.

### Result

The final analytical dataset contains 588,101 unique user-level observations across 6 analytical variables.

The cleaned dataset preserves the original experimental allocation and observed data without unnecessary transformations.